## Analiza spalania samochodów z wykorzystaniem regresji liniowej i wielomianowej
### Cel zadania
Celem ćwiczenia jest zbudowanie modeli regresyjnych przewidujących spalanie samochodu (l/100 km) na podstawie jego parametrów technicznych.
W zadaniu wykorzystasz:
- regresję liniową,  
- regresję wielomianową (kwadratową),  
- łączenie tabel (merge),  
- analizę zależności między zmiennymi,  
- ocenę jakości modeli (MSE, R²).

Zadanie pozwala zrozumieć, kiedy model liniowy jest wystarczający, a kiedy konieczne jest użycie modelu nieliniowego.

### Opis danych

**Tabela pomiarów — 150 obserwacji**
Każdy wiersz to jeden pomiar spalania dla konkretnego samochodu.

Kolumny:
- id_samochodu: identyfikator samochodu (powtarza się, bo każdy samochód ma 5 pomiarów)
- masa_kg: masa pojazdu
- moc_km: moc silnika
- opory_aero: współczynnik oporu aerodynamicznego
- spalanie_l100: zmierzona wartość spalania (zmienna objaśniana)

Liczba unikalnych samochodów: 30
Liczba pomiarów na samochód: 5
Łącznie: 150 obserwacji

Parametry techniczne zostały wygenerowane tak, aby odzwierciedlały realistyczne zależności:

- większa masa → wyższe spalanie (prawie liniowo),
- większa moc → wyższe spalanie (nieliniowo),
- większy opór aerodynamiczny → wyższe spalanie (kwadratowo).

Dzięki temu regresja liniowa będzie działać poprawnie, ale regresja kwadratowa da wyraźnie lepsze dopasowanie.

**Tabela samochody — 30 wierszy**
Zawiera unikalne identyfikatory samochodów i ich marki.

Kolumny:
- id_samochodu: klucz główny
- marka: marka pojazdu (Toyota, VW, Skoda, Ford itd.)

## Uzupełnij poszczególne komórki zadania

In [ ]:
#import bibliotek

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score
import pickle as pkl
from sklearn.pipeline import Pipeline

Pobierz dane z plików CSV (pomiary.csv i marki.csv) i wczytaj je do DataFrame'ów za pomocą biblioteki pandas. Użyj funkcji `pd.read_csv()` i dostosuj parametry, jeśli to konieczne (np. separator, kodowanie). Następnie wyświetl pierwsze kilka wierszy każdego DataFrame'a, aby upewnić się, że dane zostały poprawnie wczytane. 

**Uwaga: Przed załadowaniem danych sprawdź zawartość plików.**

```python
data = pd.read_csv(...)
print(...)
```

In [ ]:
# Wczytanie danych z plików CSV
pomiary = pd.read_csv('pomiary.csv', sep=';', encoding='utf-8')
marki = pd.read_csv('marki.csv', sep=',', encoding='utf-8')

print('Podgląd pomiary:')
print(pomiary.head())
print()
print('Podgląd marki:')
print(marki.head())

Sprawdź poprawność wszystkich danych, sknwertuj nieprawidłowe wartości (dane liczbowe zawierające przecinek zamień na kropkę). Następnie połącz oba DataFrame'y na podstawie wspólnej kolumny (np. 'marka') i utwórz nowy DataFrame zawierający wszystkie potrzebne informacje do dalszej analizy.

```python   
# Sprawdzenie poprawności danych - metoda info()
print(dane_pandas....)

# Zamiana wybranych znaków na inne (np. przecinków na kropki), konwersja danych na float dla kolumny xxx
dane_pandas['xxx'] = dane_pandas['xxx'].str.replace(..., ...).astype(float)

# Połączenie dwóch DataFrame'ów w całość (podobnie do SQL JOIN)
wynik_złączenia = pd.merge(jeden_dataframe, drugi_dataframe, on='kolumna_wspólna', how='rodzaj_złączenia')
print(wynik_złączenia.head())
```


Rodzaje złączeń:
- `inner`: Zwraca tylko te wiersze, które mają dopasowanie w obu DataFrame'ach.
- `left`: Zwraca wszystkie wiersze z lewego DataFrame'a i dopasowane wiersze z prawego DataFrame'a.
- `right`: Zwraca wszystkie wiersze z prawego DataFrame'a i dopasowane wiersze z lewego DataFrame'a.
- `outer`: Zwraca wszystkie wiersze z obu DataFrame'ów, wypełniając brakujące wartości NaN tam, gdzie nie ma dopasowania.


In [ ]:
# Sprawdzenie poprawności danych
print('Informacje o tabeli pomiary:')
print(pomiary.info())
print()
print('Informacje o tabeli marki:')
print(marki.info())

In [ ]:
# korekta i złączenie DataFrame'ów
pomiary['spalanie_l100'] = pomiary['spalanie_l100'].str.replace(',', '.', regex=False).astype(float)

# Połączenie tabel po wspólnym kluczu
merged_data = pd.merge(pomiary, marki, on='id_samochodu', how='inner')

print('Podgląd merged_data:')
print(merged_data.head())
print('Rozmiar merged_data:', merged_data.shape)

Podziel przygotowane dane na zbiór treningowy i testowy. 

Ustal zmienną objaśnianą oraz cechy wejściowe. 
**Uwaga: marka samochodu nie jest cechą numeryczną, więc nie będzie bezpośrednio używana w modelu regresyjnym.**

Następnie użyj `train_test_split`, aby np. 80% danych trafiło do treningu, a 20% do testu.

```python
X = merged_data[['cechy wejściowa1', 'cechy wejściowa2', 'cechy wejściowa3']]
y = merged_data['zmienna_objaśniana']

cechy_treningowe, cechy_testowe, y_treningowe, y_testowe = train_test_split(X, y, test_size=XXXX, random_state=XXXX)

print('Rozmiar treningu:', XXXXX.shape)
print('Rozmiar testu:', XXXXX.shape)
```

In [ ]:
# Podział na dane treningowe i testowe

X = merged_data[['masa_kg', 'moc_km', 'opory_aero']]
y = merged_data['spalanie_l100']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('Rozmiar treningu:', X_train.shape)
print('Rozmiar testu:', X_test.shape)

Wytrenuj model regresji liniowej na danych treningowych.

Użyj `LinearRegression()` i dopasuj model metodą `fit()`.
Po treningu możesz sprawdzić współczynniki modelu i wyraz wolny.

```python
model_liniowy = LinearRegression()
model_liniowy.fit(cechy_treningowe, y_treningowe)

print('Współczynniki:', model_liniowy.coef_)
print('Wyraz wolny:', model_liniowy.intercept_)
```

In [ ]:
# Trenowanie modelu liniowego
model_liniowy = LinearRegression()
model_liniowy.fit(X_train, y_train)

print('Współczynniki:', model_liniowy.coef_)
print('Wyraz wolny:', model_liniowy.intercept_)

Oceń model liniowy na zbiorze testowym.

Wyznacz prognozy (`predict`) i policz miary jakości:
- `MSE` (błąd średniokwadratowy),
- `R²` (współczynnik determinacji).

```python

y_przewidywane = model_liniowy.predict(cechy_testowe)
mse_liniowy = mean_squared_error(y_testowe, y_przewidywane)
r2_liniowy = r2_score(y_testowe, y_przewidywane)

print('MSE (liniowy):', mse_liniowy)
print('R2 (liniowy):', r2_liniowy)
```

In [ ]:
# Ocena modelu liniowego
from sklearn.metrics import mean_squared_error, r2_score

y_pred_liniowy = model_liniowy.predict(X_test)
mse_liniowy = mean_squared_error(y_test, y_pred_liniowy)
r2_liniowy = r2_score(y_test, y_pred_liniowy)

print('MSE (liniowy):', mse_liniowy)
print('R2 (liniowy):', r2_liniowy)

Wytrenuj model regresji kwadratowej (wielomianowej stopnia 2).

W tym wariancie użyj `Pipeline`, który łączy dwa kroki:
- `PolynomialFeatures(degree=2, include_bias=True/False)`,
- `LinearRegression()`


```python
model_kwadratowy = Pipeline([('poly', PolynomialFeatures(degree=2, include_bias=False)), ('model', LinearRegression()) ])

model_kwadratowy.fit(cechy_treningowe, y_treningowe)
```

In [ ]:
# Trenowanie modelu kwadratowego (wielomianowego) - stworzenie całego pipelina do trenowania i oceny

model_kwadratowy = Pipeline([("poly", PolynomialFeatures(degree=2, include_bias=False)), ("model", LinearRegression())])

_ = model_kwadratowy.fit(X_train, y_train)


Oceń model kwadratowy na danych testowych i policz te same metryki (`MSE`, `R²`).

Ponieważ model jest zapisany jako `Pipeline`, podajesz do `predict()` bezpośrednio `cechy_treningowe`, a transformacja wielomianowa wykona się automatycznie w środku.

```python
y_przewidywane = model_kwadratowy.predict(cechy_treningowe)
mse_kwadratowy = mean_squared_error(y_testowe, y_przewidywane)
r2_kwadratowy = r2_score(y_testowe, y_przewidywane)

print('MSE (kwadratowy):', mse_kwadratowy)
print('R2 (kwadratowy):', r2_kwadratowy)
```

In [ ]:
# Ocena modelu kwadratowego
y_pred_kwadratowy = model_kwadratowy.predict(X_test)
mse_kwadratowy = mean_squared_error(y_test, y_pred_kwadratowy)
r2_kwadratowy = r2_score(y_test, y_pred_kwadratowy)

print('MSE (kwadratowy):', mse_kwadratowy)
print('R2 (kwadratowy):', r2_kwadratowy)

Porównaj oba modele i wybierz lepszy.

Wybór oprzyj o metryki:
- niższe `MSE` oznacza mniejszy błąd,
- wyższe `R²` oznacza lepsze dopasowanie.

Oceń
- który model wygrał,
- czy poprawa jakości była duża czy mała,
- dlaczego taki model jest lepszy dla tych danych.

```python
if (...) and (...):
    lepszy_model = 'kwadratowy'
else:
    lepszy_model = 'liniowy'

print('Lepszy model:', lepszy_model)
```

In [ ]:
# Porównanie modeli i wybór lepszego
if (r2_kwadratowy > r2_liniowy) and (mse_kwadratowy < mse_liniowy):
    lepszy_model = 'kwadratowy'
else:
    lepszy_model = 'liniowy'

print('Lepszy model:', lepszy_model)
print('MSE liniowy:', mse_liniowy, '| R2 liniowy:', r2_liniowy)
print('MSE kwadratowy:', mse_kwadratowy, '| R2 kwadratowy:', r2_kwadratowy)

Zapisz wybrany model do pliku `pickle`, aby można było użyć go później bez ponownego trenowania.

W tym notebooku zapisujesz finalnie jeden obiekt modelu do pliku `model.pkl`.

```python
model = model_kwadratowy #lepszy model

with open('model.pkl', 'wb') as f:
    pkl.dump(model, f)

print('Model zapisano do pliku: model.pkl')
```

In [ ]:
# Eksport wybranego modelu do pliku

model = model_kwadratowy

with open('model.pkl', 'wb') as f:
    pkl.dump(model, f)

print('Model zapisano do pliku: model.pkl')